In [6]:
import pandas as pd   # data handling
import numpy as np    # numeric helpers

# Load the CSV (change path if needed)
df = pd.read_csv('/workspaces/Project_01/video_game_analysis/notebooks/Fc110542_Aashinshana/Video_Games_Sales_as_at_22_Dec_2016.csv')  

# Show first 10 rows so we can see column names and sample data
print("HEAD (first 10 rows):")
print(df.head(10))

# Show concise info: dtypes and non-null counts
print("\nINFO():")
print(df.info())

# Show basic stats for numeric columns
print("\nDESCRIBE() for numeric columns:")
print(df.describe(include=[np.number]).T)  # transpose for readability

# Show basic stats for object (categorical/text) columns
print("\nDESCRIBE() for object columns (unique counts):")
print(df.describe(include=[object]).T)

# Check missing values per column (counts and percent)
print("\nMISSING VALUES (counts and percentage):")
missing_counts = df.isna().sum()
missing_pct = (missing_counts / len(df)) * 100
print(pd.concat([missing_counts, missing_pct.rename("missing_pct")], axis=1))

# Check duplicate rows count (exact duplicates)
print("\nDUPLICATE ROWS (exact duplicates):")
print("duplicate_count =", df.duplicated().sum())

# Check unique cardinalities for key categorical columns
categorical_cols = ['Platform', 'Genre', 'Publisher']  # common categorical cols in this dataset
print("\nUNIQUE COUNTS for categorical columns:")
for col in categorical_cols:
    if col in df.columns:
        print(f"{col}: unique={df[col].nunique()}  -  top 10 values:")
        print(df[col].value_counts().head(10))
    else:
        print(f"{col}: NOT FOUND in dataframe")

# Quick look at year column (if present) to spot weird years / types
if 'Year_of_Release' in df.columns:
    print("\nYEAR column unique values and stats:")
    # show dtype, nulls, min/max
    print("dtype:", df['Year_of_Release'].dtype, " nulls:", df['Year_of_Release'].isna().sum())
    # try to coerce to numeric (some datasets have 'N/A' or 'TBD')
    years_numeric = pd.to_numeric(df['Year_of_Release'], errors='coerce')
    print("min year (coerced):", int(years_numeric.min()) if years_numeric.notna().any() else "NA")
    print("max year (coerced):", int(years_numeric.max()) if years_numeric.notna().any() else "NA")
    print("year value_counts (top 10):")
    print(years_numeric.value_counts().sort_index().head(20))

# Inspect target distribution quickly (Global_Sales)
if 'Global_Sales' in df.columns:
    print("\nTARGET (Global_Sales) basic stats and skewness:")
    print(df['Global_Sales'].describe())
    print("skewness:", df['Global_Sales'].skew(), "  kurtosis:", df['Global_Sales'].kurtosis())
    # show top rows of largest sales
    print("\nTop 10 games by Global_Sales:")
    print(df[['Name','Platform','Year_of_Release','Global_Sales']].sort_values('Global_Sales', ascending=False).head(10))
else:
    print("\nWARNING: 'Global_Sales' column not found. Please confirm the target column name.")


HEAD (first 10 rows):
                        Name Platform  Year_of_Release         Genre  \
0                 Wii Sports      Wii           2006.0        Sports   
1          Super Mario Bros.      NES           1985.0      Platform   
2             Mario Kart Wii      Wii           2008.0        Racing   
3          Wii Sports Resort      Wii           2009.0        Sports   
4   Pokemon Red/Pokemon Blue       GB           1996.0  Role-Playing   
5                     Tetris       GB           1989.0        Puzzle   
6      New Super Mario Bros.       DS           2006.0      Platform   
7                   Wii Play      Wii           2006.0          Misc   
8  New Super Mario Bros. Wii      Wii           2009.0      Platform   
9                  Duck Hunt      NES           1984.0       Shooter   

  Publisher  NA_Sales  EU_Sales  JP_Sales  Other_Sales  Global_Sales  \
0  Nintendo     41.36     28.96      3.77         8.45         82.53   
1  Nintendo     29.08      3.58      6.81

In [7]:
# Drop columns with too many missing values (they'll hurt SVR)
cols_to_drop = ['Critic_Score', 'Critic_Count', 'User_Count', 'Developer', 'Rating']
df = df.drop(columns=cols_to_drop)

# Handle 'User_Score' - replace 'tbd' with NaN, convert to numeric
df['User_Score'] = pd.to_numeric(df['User_Score'].replace('tbd', np.nan), errors='coerce')

# Fill missing Publisher with "Unknown"
df['Publisher'] = df['Publisher'].fillna('Unknown')

# Fill missing Year_of_Release with median (and convert to int)
df['Year_of_Release'] = df['Year_of_Release'].fillna(df['Year_of_Release'].median()).astype(int)

# Drop rows where 'Name' or 'Genre' are missing
df = df.dropna(subset=['Name', 'Genre'])

# Check how many rows remain
print("Rows remaining after cleaning:", len(df))

# Confirm no major missing data remains
print("\nRemaining missing values:")
print(df.isna().sum())

# Check data types now
print("\nUpdated dtypes:")
print(df.dtypes.head(10))

# Preview top rows
print("\nCleaned data sample:")
print(df.head(5))


Rows remaining after cleaning: 16717

Remaining missing values:
Name                  0
Platform              0
Year_of_Release       0
Genre                 0
Publisher             0
NA_Sales              0
EU_Sales              0
JP_Sales              0
Other_Sales           0
Global_Sales          0
User_Score         9127
dtype: int64

Updated dtypes:
Name                object
Platform            object
Year_of_Release      int64
Genre               object
Publisher           object
NA_Sales           float64
EU_Sales           float64
JP_Sales           float64
Other_Sales        float64
Global_Sales       float64
dtype: object

Cleaned data sample:
                       Name Platform  Year_of_Release         Genre Publisher  \
0                Wii Sports      Wii             2006        Sports  Nintendo   
1         Super Mario Bros.      NES             1985      Platform  Nintendo   
2            Mario Kart Wii      Wii             2008        Racing  Nintendo   
3         Wi